# A3 Step 3 -- CLIP visual-retrieval fallback: embed all 1040 content pages

Scoped visual fallback (DECISION D2, `plan_a3.md` Sec.5): embeds every page image with
`openai/clip-vit-base-patch32` so the two real A2 failures -- the 53 pages with no text
chunk at all, and the under-transcribed-table case -- both become recoverable through
`Retriever._augment_with_visual_fallback()`, without building a full separate ColPali
system (the repo's fixed structure has no home for a new top-level module).

Self-contained on purpose: does not clone the repo, only needs the
`mathscholar-a3-pages` dataset (the exact 1040 pages `loader.load_pages()` returns,
matching `data/ocr/`'s own page count) mounted as input. Output,
`image_embed_cache.npz`, gets downloaded and placed at `data/index/image_embed_cache.npz`
locally -- gitignored, same policy as the rest of `data/index/`.

In [ ]:
import time
import zipfile
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

# Pinned to the exact commit resolved for openai/clip-vit-base-patch32 -- matches
# src/doc_agent/retrieval/retriever.py's own pin exactly (bandit B615 fix, same
# precedent as A2's TATR revision pin in vision/layout.py). Keep these two in sync.
MODEL_NAME = "openai/clip-vit-base-patch32"
REVISION = "3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268"

OUTPUT_PATH = Path("/kaggle/working/image_embed_cache.npz")
BATCH_SIZE = 32

def _pick_device() -> str:
    """torch.cuda.is_available() only checks a driver is present -- it does NOT check
    the GPU's compute capability is actually supported by this PyTorch build. Kaggle
    can assign an older GPU (seen in practice: a Tesla P100, sm_60) that a newer PyTorch
    wheel (built for sm_70+) can detect but not actually run a kernel on -- that combination
    reports "cuda available" and then hard-crashes on the first real op. Try a real,
    trivial CUDA op and fall back to CPU on failure, rather than trusting availability
    alone. CLIP-B/32 over 1040 pages is small enough that CPU is a tolerable fallback,
    not a blocker (this fallback is optional per plan_a3.md Step 3, not required)."""
    if not torch.cuda.is_available():
        return "cpu"
    try:
        _probe = torch.zeros(1, device="cuda")
        _ = _probe + 1
        return "cuda"
    except RuntimeError as exc:
        print(f"CUDA reports available but a real op failed ({exc}) -- using CPU instead")
        return "cpu"


device = _pick_device()
print(f"device: {device}")

t0 = time.time()
model = CLIPModel.from_pretrained(MODEL_NAME, revision=REVISION).to(device).eval()
processor = CLIPProcessor.from_pretrained(MODEL_NAME, revision=REVISION)
print(f"model loaded in {time.time()-t0:.1f}s")

# Three prior runs each guessed a mount layout and each guess was wrong -- first
# /kaggle/input/mathscholar-a3-pages/ directly, then one level of subdirectories under
# /kaggle/input/, and the real one turned out to be /kaggle/input/datasets/<slug>/
# (a newer Kaggle layout that groups dataset mounts under an extra "datasets" dir,
# different from the classic layout assumed at the start). Stop guessing depth
# entirely: search recursively, arbitrarily deep, for the actual files.
print("recursive listing of /kaggle/input/ (first 20 entries):")
all_entries = list(Path("/kaggle/input").rglob("*"))
for p in all_entries[:20]:
    print(" ", p)
print(f"  ... {len(all_entries)} entries total")

image_files = sorted(Path("/kaggle/input").rglob("as_p*.png"))
if not image_files:
    zips = sorted(Path("/kaggle/input").rglob("*.zip"))
    if zips:
        extract_dir = Path("/kaggle/working/pages_extracted")
        extract_dir.mkdir(parents=True, exist_ok=True)
        print(f"no loose PNGs anywhere under /kaggle/input/ -- extracting {zips[0]}")
        with zipfile.ZipFile(zips[0]) as zf:
            zf.extractall(extract_dir)
        image_files = sorted(extract_dir.glob("as_p*.png"))

assert image_files, (
    "no as_p*.png found under any /kaggle/input/ subdirectory, and no zip to extract -- "
    "see the directory listing printed above for what's actually mounted"
)

print(f"found {len(image_files)} page images")
assert len(image_files) == 1040, (
    f"expected exactly 1040 content pages (loader.load_pages()'s own count, A2/A3 "
    f"convention) -- got {len(image_files)}. Check the mounted dataset."
)

page_ids: list[str] = []
vector_batches: list[np.ndarray] = []

t0 = time.time()
with torch.no_grad():
    for i in range(0, len(image_files), BATCH_SIZE):
        batch_files = image_files[i : i + BATCH_SIZE]
        images = [Image.open(f).convert("RGB") for f in batch_files]
        inputs = processor(images=images, return_tensors="pt").to(device)
        # NOT model.get_image_features(**inputs) -- confirmed locally (transformers
        # 4.44.2) that call returns a plain tensor via vision_model -> visual_projection,
        # but Kaggle's pre-installed transformers version returned the raw
        # BaseModelOutputWithPooling instead (a real version-dependent API break, caught
        # by a run that actually got this far rather than assumed compatible). Replicate
        # get_image_features()'s own internals directly -- vision_model + visual_projection
        # are stable core submodules, not a convenience wrapper that can silently change
        # its return contract between transformers versions.
        vision_out = model.vision_model(pixel_values=inputs["pixel_values"])
        pooled_output = vision_out[1]  # positional, not .pooler_output -- works across versions
        feats = model.visual_projection(pooled_output)
        feats = feats / feats.norm(dim=-1, keepdim=True)  # L2-normalise for cosine
        vector_batches.append(feats.cpu().numpy().astype(np.float32))
        page_ids.extend(f.stem for f in batch_files)
        if (i // BATCH_SIZE) % 5 == 0:
            elapsed = time.time() - t0
            print(f"  {i + len(batch_files)}/{len(image_files)}  ({elapsed:.1f}s elapsed)")

vectors = np.concatenate(vector_batches, axis=0)
assert vectors.shape[0] == len(page_ids) == 1040
print(f"done in {time.time()-t0:.1f}s -- final shape {vectors.shape}")

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
np.savez(OUTPUT_PATH, page_ids=np.array(page_ids), vectors=vectors)
size_mb = OUTPUT_PATH.stat().st_size / 1e6
print(f"saved {OUTPUT_PATH} ({size_mb:.1f} MB)")

# Sanity check: a page should be most similar to itself, and cosine range should look real
sample_scores = vectors[:20] @ vectors[:20].T
print("self-similarity diagonal (should all be ~1.0):", np.diag(sample_scores)[:5])
print("off-diagonal range:", sample_scores[~np.eye(20, dtype=bool)].min(),
      "to", sample_scores[~np.eye(20, dtype=bool)].max())
